In [ ]:
import requests
from bs4 import BeautifulSoup
from pydantic import BaseModel
from typing import Optional
from openai import OpenAI
from google.colab import userdata

# 1. Modelo estruturado
class CartuchoAnuncio(BaseModel):
    titulo: Optional[str]
    marca: Optional[str]
    modelo: Optional[str]
    preco: Optional[float]
    cor: Optional[str]
    qualidade_descricao: Optional[str]
    quantidade_reviews: Optional[int]
    avaliação: Optional[float]
    quantidade_fotos: Optional[int]

# 2. URL do anúncio no Mercado Livre
URL = "https://www.mercadolivre.com.br/cartucho-hp-667-preto-2376-2776-6476/p/MLB22022306#reco_item_pos=1&reco_backend=item_decorator&reco_backend_type=function&reco_client=home_items-decorator-legacy&reco_id=0258d252-7cb0-4260-966c-771c5e56ac75&reco_model=&c_id=/home/navigation-recommendations-seed/element&c_uid=7664c82d-c786-4169-880f-2774d28baf3e&da_id=navigation&da_position=1&id_origin=/home/dynamic_access&da_sort_algorithm=ranker"  # troque pela URL real

# 3. Fazer requisição HTTP para obter o HTML
headers = {
    "User-Agent": "Mozilla/5.0"
}
response = requests.get(URL, headers=headers)
html = response.text

# 4. Parsear HTML com BeautifulSoup
soup = BeautifulSoup(html, "html.parser")
titulo = soup.title.string if soup.title else None
texto = soup.get_text(separator="\n")

# 5. Chamar OpenAI com resposta estruturada
client = OpenAI(api_key=userdata.get('OPENAI_API_KEY')) # Certifique-se de ter OPENAI_API_KEY configurada

completion = client.beta.chat.completions.parse(
    model="gpt-4o",
    messages=[
        {
            "role": "system",
            "content": (
                "Extraia informações estruturadas sobre o anúncio de cartucho. "
                "Os campos esperados são: titulo, marca, modelo, preco, cor, "
                "qualidade_descricao, quantidade_reviews, avaliação, quantidade_fotos."
            )
        },
        {
            "role": "user",
            "content": texto
        }
    ],
    response_format=CartuchoAnuncio
)

# 6. Resultado
resultado = completion.choices[0].message.parsed
print(resultado)

# para visualizar resultados de forma separada:
print(f"Titulo: {resultado.titulo}")
print(f"Marca: {resultado.marca}")
print(f"Modelo: {resultado.modelo}")
print(f"Preço: {resultado.preco}")

titulo='Cartucho HP 667 Preto 2376 2776 6476' marca='HP' modelo='667' preco=66.9 cor='Preto' qualidade_descricao='Imprima todas as fotos e documentos de alta qualidade. Desempenho em alta velocidade as tintas originais HP são elaboradas para ajudar você a imprimir rapidamente sem abrir mão da qualidade.' quantidade_reviews=1933 avaliação=4.7 quantidade_fotos=7
Titulo: Cartucho HP 667 Preto 2376 2776 6476
Marca: HP
Modelo: 667
Preço: 66.9
